In [ ]:
import pandas as pd

df = pd.read_csv("merged_reviews.csv")
print(f"Reviews loaded: {len(df)}")
print(df['game_name'].value_counts())

Reviews loaded: 30104
game_name
helldivers2             10077
monster_hunter_wilds    10061
borderlands4             9966
Name: count, dtype: int64


In [ ]:
from transformers import pipeline

print("Loading model... this takes about 2 minutes")
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0  # uses GPU in Colab, remove this line if running locally
)
print("Model loaded!")

In [ ]:
df = df.dropna(subset=["cleaned_review"])

In [ ]:
print(df["cleaned_review"].isna().sum())

0


In [ ]:
# Filter to relevant reviews only
df = df[
    (df['voted_up'] == False) |
    (df['cleaned_review'].str.lower().str.contains(
        'crash|bug|glitch|freeze|error|fps|lag|stutter|fix|broken|disconnect|corrupt|performance|issue|problem',
        na=False
    ))
]
print(f"Reviews to classify after filter: {len(df)}")

# Reset index after filtering
df = df.reset_index(drop=True)

Reviews to classify after filter: 17689


In [ ]:
candidate_labels = [
    "specific technical issue such as game crash, performance problem, bug, or error",
    "general opinion or emotional frustration without specific technical details"
]

predictions = []

# Run in batches of 32 to avoid memory issues
batch_size = 32
total = len(df)

for i in range(0, total, batch_size):
    batch = df['cleaned_review'].iloc[i:i+batch_size].tolist()

    results = classifier(
        batch,
        candidate_labels=candidate_labels,
        multi_label=False
    )

    for result in results:
        # Top label is whichever scored higher
        top_label = result['labels'][0]
        score = result['scores'][0]

        if top_label == "specific technical issue such as game crash, performance problem, bug, or error":
          predictions.append({'label': 'escalation', 'confidence': round(score, 3)})
        else:
          predictions.append({'label': 'venting', 'confidence': round(score, 3)})

    print(f"Classified {min(i+batch_size, total)}/{total} reviews...")

print("Classification complete!")

In [ ]:
# Check raw predictions before threshold
pred_df = pd.DataFrame(predictions)
df['label_raw'] = pred_df['label'].values
df['confidence'] = pred_df['confidence'].values

# Distribution of confidence scores for escalations
escalations_raw = df[df['label_raw'] == 'escalation']
print(f"Total raw escalations before threshold: {len(escalations_raw)}")
print(f"\nConfidence distribution:")
print(escalations_raw['confidence'].describe())

# See how many survive at different thresholds
for threshold in [0.50, 0.55, 0.60, 0.65, 0.70]:
    count = len(escalations_raw[escalations_raw['confidence'] >= threshold])
    print(f"Threshold {threshold}: {count} escalations survive")

Total raw escalations before threshold: 1652

Confidence distribution:
count    1652.000000
mean        0.631122
std         0.102771
min         0.500000
25%         0.546000
50%         0.606000
75%         0.694000
max         0.986000
Name: confidence, dtype: float64
Threshold 0.5: 1652 escalations survive
Threshold 0.55: 1196 escalations survive
Threshold 0.6: 871 escalations survive
Threshold 0.65: 611 escalations survive
Threshold 0.7: 393 escalations survive


In [ ]:
df_results = df.copy()
df_results['label'] = pred_df['label'].values
df_results['confidence'] = pred_df['confidence'].values

# Only look at escalations above your threshold
high_conf_escalations = df_results[
    (df_results['label'] == 'escalation') &
    (df_results['confidence'] >= 0.65)
]

print("=== SAMPLE ESCALATIONS ===")
for _, row in high_conf_escalations.sample(5).iterrows():
    print(f"\n[{row['game_name']}] Confidence: {row['confidence']}")
    print(row['cleaned_review'][:200])

=== SAMPLE ESCALATIONS ===

[borderlands4] Confidence: 0.694
My 13900KS + RTX5090 crashes very often. Usually it is at certain spots of the game. Latest game update and nvidia drivers helped a little but it is still happening.

[helldivers2] Confidence: 0.651
Game crash, enemies glitching through, Meaningless Progression.    Developers are creating problems and using warbonds to solve them, and locking out meaningful progression + content hidden in the gam

[helldivers2] Confidence: 0.707
crashes alot

[borderlands4] Confidence: 0.774
Performance issues.  Compiling Shaders at many loading screens.  Endgame is farming the same bosses that die extremely quickly at max difficulty for rare drops to kill the same bosses slightly faster.

[monster_hunter_wilds] Confidence: 0.761
i keep coming back every few months to see if theyve fixed the issue with amd gpus that causes the game to keep crashing and just as i expected they havent done anything


In [ ]:
# Apply final threshold
df_results = df.copy()
df_results['label'] = pred_df['label'].values
df_results['confidence'] = pred_df['confidence'].values

df_results['final_label'] = df_results.apply(
    lambda row: row['label'] if row['confidence'] >= 0.65 else 'uncertain',
    axis=1
)

# Save full results
df_results.to_csv('review_labels.csv', index=False)

Reloading the dataframes again to prevent having to classify all the reviews again.

In [ ]:
df = pd.read_csv('review_labels.csv')

In [ ]:
pred_df = pd.read_csv('pred_df.csv')

In [ ]:
# Look at 5 escalations and 5 ventings to verify quality
print("=== SAMPLE ESCALATIONS ===")
escalations = df[df['label'] == 'escalation'].sample(5)
for _, row in escalations.iterrows():
    print(f"\n[{row['game_name']}] Confidence: {row['confidence']}")
    print(row['cleaned_review'][:200])

print("\n=== SAMPLE VENTINGS ===")
ventings = df[df['label'] == 'venting'].sample(5)
for _, row in ventings.iterrows():
    print(f"\n[{row['game_name']}] Confidence: {row['confidence']}")
    print(row['cleaned_review'][:200])

=== SAMPLE ESCALATIONS ===

[borderlands4] Confidence: 0.58
it barely runs idk

[helldivers2] Confidence: 0.635
Game unplayable, enemies spawn from behind walls and mountains. hit boxes are horrible. Devs cant beat the game on highest difficulty, and do not play there game.

[helldivers2] Confidence: 0.749
Game is full of bugs, bots and squidwards

[borderlands4] Confidence: 0.511
just doing this review for the badge.

[helldivers2] Confidence: 0.508
BRUN LOOT MURDER

=== SAMPLE VENTINGS ===

[monster_hunter_wilds] Confidence: 0.674
The Hunter community is great.

[helldivers2] Confidence: 0.726
ok if you are going to play this game just dont fight the illuminate and you will have some fun

[helldivers2] Confidence: 0.786
Developers largely do not value the feedback and wishes of players. Continually move to make the playstyles added by new DLC material 'fun', whilst either reducing the viability of older content or ju

[monster_hunter_wilds] Confidence: 0.909
Worst monster hunter game

In [ ]:
import pandas as pd

df = pd.read_csv('review_labels.csv')

# Keep only what BigQuery needs
upload_df = df[['recommendationid', 'game_name', 'final_label', 'confidence']].copy()

# Rename for clarity
upload_df = upload_df.rename(columns={'final_label': 'label'})

print(upload_df['label'].value_counts())
print(upload_df.shape)

# Export as JSONL
upload_df.to_json('review_labels.json', orient='records', lines=True, force_ascii=False)

label
venting       14109
uncertain      2969
escalation      611
Name: count, dtype: int64
(17689, 4)
